# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [ ]:
# My lane: Content Refresh Prioritization.
# Decision it improves: "Which published pages should an editor refresh FIRST?"
#
# Framed as RANKING (with a classification view underneath):
#   - Ranking: score every page by how likely it is to be losing traffic, then
#     work the top of the list. The editor only has time for a handful per week,
#     so an ordered queue is what actually gets used.
#   - The same signal can be read as CLASSIFICATION ("is this page declining:
#     yes/no?") using the observed trend_direction == "down" label, which is how
#     I get a defensible metric below.
#
# Why not clustering? I have an observed outcome (traffic went down), so a
# supervised task is honest and checkable. Clustering would only describe
# "kinds of pages", not tell the editor what to do first.

task_type = "ranking (scored priority queue), evaluated as binary classification"
print("Lane:", "content refresh prioritization")
print("Task type:", task_type)

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [ ]:
import pandas as pd
from pathlib import Path

# Load the anonymized slice. Try the repo path first, fall back to a local copy.
CANDIDATES = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]
csv_path = next((p for p in CANDIDATES if p.exists()), CANDIDATES[0])
df = pd.read_csv(csv_path)

# TARGET: is this page losing traffic? -> declining = (trend_direction == "down")
# This is an OBSERVED outcome. trend_direction / trend_pct are measured by
# comparing a later 30-day window against the prior 30-day window
# (impressions_last_30d / sessions_last_30d vs impressions_prev_30d / ...),
# not set by a human rule. So the model learns the world, not someone's if-statement.
df["declining"] = (df["trend_direction"] == "down").astype(int)

print("Rows:", len(df))
print(df["trend_direction"].value_counts(dropna=False))
print("Base rate (declining):", round(df["declining"].mean(), 3))

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [ ]:
# The editor can only refresh a limited number of pages each cycle, so what
# matters is: of the pages we put at the TOP of the queue, how many are really
# declining? That is precision@K.
#
# I report a naive baseline: sort by trend_pct (most negative first) and check
# precision in the top K. A useful model must beat this and beat the base rate.

import numpy as np

base_rate = df["declining"].mean()

ranked = df.sort_values("trend_pct", ascending=True)  # most negative % first
for K in [50, 100, 500]:
    topK = ranked.head(K)
    p_at_k = topK["declining"].mean()
    print(f"precision@{K:<4} = {p_at_k:.3f}   (base rate = {base_rate:.3f})")

# "Good" target I will defend: precision@100 >= 0.85 while clearly beating the
# base rate. Editors trust a queue where ~9 of every 10 top items are real
# declines; below that the list gets ignored.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
# One row = one PIECE OF PUBLISHED CONTENT (a page), identified by content_id,
# with its 90-day and 30-day traffic, on-page attributes, and trend labels.
unit_cols = [
    "content_id", "content_type", "main_intent",
    "sessions_last_30d", "sessions_prev_30d",
    "avg_position", "position_tier",
    "days_since_last_update", "freshness_tier",
    "trend_pct", "trend_direction", "declining",
]
print("Unit of analysis: one row = one published page (content_id)")
print("Shape:", df.shape)
df[unit_cols].head(10)

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
# A single hand-written rule (e.g. "flag it if sessions dropped > 30%") misses a
# lot, because decline shows up through MANY tangled signals that interact:
#   - position (avg_position, position_tier) and impression volume
#   - intent + content_type (transactional vs informational pages age differently)
#   - freshness (days_since_last_update) and content_age_days
#   - engagement (ctr, engagement_rate, scroll_rate) and the split of ai_traffic_pct
#
# Quick evidence: the naive "big % drop" rule is noisy vs the observed label.
naive_flag = (df["trend_pct"] < -20)
tp = int(((naive_flag == 1) & (df["declining"] == 1)).sum())
fp = int(((naive_flag == 1) & (df["declining"] == 0)).sum())
fn = int(((naive_flag == 0) & (df["declining"] == 1)).sum())
precision = tp / (tp + fp) if (tp + fp) else float("nan")
recall = tp / (tp + fn) if (tp + fn) else float("nan")
print(f"Fixed rule (trend_pct < -20): precision={precision:.3f}, recall={recall:.3f}")

# Group check: decline rate differs by tier, so no single threshold fits all.
print()
print(df.groupby("position_tier")["declining"].mean().round(3).sort_values(ascending=False))

# Conclusion (careful wording): these are OBSERVED, DIRECTIONAL signals meant as
# DECISION-SUPPORT for prioritizing refresh work — not a causal claim. ML earns
# its place because the boundary between "declining" and "fine" bends with
# position, intent, freshness and engagement all at once, which a fixed
# if-statement can't track without a growing pile of exceptions.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.